# MLS/WACCM chemistry and dehydration

Raw inputs are read from `PAPER1_ARCHIVE_ROOT`; preprocessing products are read from `PAPER1_PREPROCESSED_ROOT`; new diagnostics are written to `PAPER1_DERIVED_ROOT` (default: repository-local `work/`). Source files are never modified.


## Shared no-leap chemistry functions

Inputs: daily MLS `value`/`nvalues` and staged WACCM chemistry/temperature. Outputs: strict unit conversion, polar interpolation, and anomaly helpers. Method: all fields are unsmoothed, 60--82N, and 1--100 hPa; MLS must have unique calendar dates, each of 365 no-leap month-days exactly 15 times in the 2004/05--2018/19 baseline and exactly once in the 2019/20 event.


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

def discover_diagnostic_directory():
    candidates = (
        Path.cwd(), Path.cwd() / "analysis",
        Path.cwd() / "Paper1" / "analysis",
    )
    for candidate in candidates:
        if (candidate / "lib" / "workflow_io.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Cannot locate Paper1/analysis/lib from the current working directory"
    )

NOTEBOOK_DIR = discover_diagnostic_directory()
LIB = NOTEBOOK_DIR / "lib"
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))

from workflow_io import (
    PRODUCT_VERSION, archive_root, derived_root, preprocessed_root, product_path,
    write_csv_atomic, write_netcdf_atomic,
)

ARCHIVE_ROOT = archive_root()
PREPROCESSED_ROOT = preprocessed_root()
DERIVED_ROOT = derived_root()
MARCH_HINDCAST_ROOT = Path(os.environ.get(
    "PAPER1_MARCH_HINDCAST_SOURCE",
    ""
    "",
))
OVERWRITE = os.environ.get("PAPER1_OVERWRITE_STAGING", "0") == "1"
print("read-only archive root:", ARCHIVE_ROOT)
print("preprocessed staging input root:", PREPROCESSED_ROOT)
print("read-only March hindcast root:", MARCH_HINDCAST_ROOT)
print("staging output root:", DERIVED_ROOT)
print("diagnostic notebook directory:", NOTEBOOK_DIR)

from paper1_diagnostics import (
    cosine_latitude_mean, date_int, hybrid_mid_pressure, log_pressure_interpolate,
    noleap_index, parse_year, select_latitude,
)

CHEM_PLEV = np.asarray([1, 2, 3, 5, 7, 10, 15, 20, 30, 50, 70, 100], dtype=float)
OCTOBER_START = 273
NOLEAP_MONTH_LENGTHS = (31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31)
REQUIRED_MONTH_DAYS = tuple(
    f"{month:02d}-{day:02d}"
    for month, length in enumerate(NOLEAP_MONTH_LENGTHS, start=1)
    for day in range(1, length + 1)
)

def require_mls_daily_calendar(dates, *, expected_per_month_day, context):
    """Require unique dates and an exact no-leap month-day multiplicity."""

    index = pd.DatetimeIndex(pd.to_datetime(np.asarray(dates)))
    if index.hasnans:
        raise RuntimeError(f"{context}: calendar contains an unparseable timestamp")
    calendar_dates = index.strftime("%Y-%m-%d").to_numpy()
    if np.unique(calendar_dates).size != calendar_dates.size:
        duplicate_dates = pd.Series(calendar_dates)[
            pd.Series(calendar_dates).duplicated(keep=False)
        ].unique()
        raise RuntimeError(
            f"{context}: duplicate calendar date(s): {','.join(duplicate_dates[:5])}"
        )
    if np.any((index.month == 2) & (index.day == 29)):
        raise RuntimeError(f"{context}: Feb 29 must be removed before calendar validation")
    month_days = index.strftime("%m-%d").to_numpy()
    counts = pd.Series(month_days).value_counts().reindex(REQUIRED_MONTH_DAYS, fill_value=0)
    unexpected = sorted(set(month_days) - set(REQUIRED_MONTH_DAYS))
    bad = counts[counts != int(expected_per_month_day)]
    if unexpected or not bad.empty:
        details = ",".join(f"{key}:{int(value)}" for key, value in bad.head(8).items())
        raise RuntimeError(
            f"{context}: every no-leap month-day must occur exactly "
            f"{expected_per_month_day} time(s); mismatches={details or 'unexpected keys'}"
        )
    expected_size = len(REQUIRED_MONTH_DAYS) * int(expected_per_month_day)
    if index.size != expected_size:
        raise RuntimeError(f"{context}: expected {expected_size} unique daily records; found {index.size}")
    return month_days

def convert_mole_fraction(field, *, source_kind, target_unit, context):
    """Strictly convert the two audited source conventions to ppbv/ppmv."""

    original_units = str(field.attrs.get("units", "")).strip()
    normalized = "".join(character for character in original_units.lower() if character.isalnum())
    expected = {"MLS": "vmr", "WACCM": "molmol"}[source_kind]
    if normalized != expected:
        raise ValueError(
            f"{context}: expected audited {source_kind} units {expected!r}; "
            f"found {original_units!r}"
        )
    factor = {"ppbv": 1.0e9, "ppmv": 1.0e6}[target_unit]
    converted = field * factor
    converted.attrs.update(
        units=target_unit, source_units=original_units,
        conversion=f"{source_kind} {original_units} x {factor:g} -> {target_unit}",
    )
    return converted

def mls_event_anomaly(variable, group, target_unit):
    root = ARCHIVE_ROOT / "MLS" / "Level3_Zonal_v5" / variable
    by_year = {}
    for path in sorted(root.glob(f"MLS-Aura_L3DZ-{variable}_v05-*_*.nc")):
        by_year[int(path.stem.rsplit("_", 1)[-1])] = path
    required = set(range(2004, 2021))
    if set(by_year) & required != required:
        raise FileNotFoundError(f"MLS {variable}: annual files 2004--2020 are required")
    records, original_units, conversions = [], set(), set()
    for year in sorted(required):
        with xr.open_dataset(by_year[year], group=group, decode_times=True, engine="netcdf4") as ds:
            value = convert_mole_fraction(
                ds["value"], source_kind="MLS", target_unit=target_unit,
                context=f"MLS {variable} {year}",
            )
            original_units.add(value.attrs["source_units"])
            conversions.add(value.attrs["conversion"])
            value = select_latitude(value, 60.0, 82.0)
            nvalues = select_latitude(ds["nvalues"], 60.0, 82.0)
            pressure_name = "lev" if "lev" in value.coords else "plev"
            value = value.where(nvalues > 0).where(
                (value[pressure_name] >= 1.0) & (value[pressure_name] <= 100.0), drop=True
            )
            polar = cosine_latitude_mean(value, 60.0, 82.0).transpose("time", pressure_name).load()
            records.append(polar.rename({pressure_name: "plev"}))
    all_values = xr.concat(records, dim="time").sortby("time")
    dates = pd.DatetimeIndex(pd.to_datetime(all_values.time.values))
    keep = ~((dates.month == 2) & (dates.day == 29))
    all_values = all_values.isel(time=np.flatnonzero(keep))
    dates = dates[keep]
    month_day = dates.strftime("%m-%d").to_numpy()
    all_values = all_values.assign_coords(month_day=("time", month_day))
    baseline = all_values.sel(time=slice("2004-10-01", "2019-09-30"))
    event = all_values.sel(time=slice("2019-10-01", "2020-09-30"))
    require_mls_daily_calendar(
        baseline.time.values, expected_per_month_day=15,
        context=f"MLS {variable} 2004/05--2018/19 baseline",
    )
    require_mls_daily_calendar(
        event.time.values, expected_per_month_day=1,
        context=f"MLS {variable} 2019/20 event",
    )
    climatology = baseline.groupby("month_day").mean("time", skipna=True)
    selected = climatology.sel(month_day=xr.DataArray(event.month_day.values, dims="time"))
    selected = selected.drop_vars("month_day").assign_coords(time=event.time)
    if len(original_units) != 1 or len(conversions) != 1:
        raise RuntimeError(f"MLS {variable}: source units/conversion changed across annual files")
    anomaly = (event.drop_vars("month_day") - selected).transpose("time", "plev").load()
    anomaly.attrs.update(
        units=target_unit, source_units=next(iter(original_units)),
        conversion=next(iter(conversions)),
        calendar_gate=(
            "unique calendar dates; all 365 no-leap month-days exactly 15 times "
            "in baseline and exactly once in event"
        ),
        baseline_month_day_count=15, event_month_day_count=1,
    )
    return anomaly

def waccm_polar(path, variable, target_unit):
    with xr.open_dataset(path, decode_times=True, chunks={"time": 12}) as ds:
        source = convert_mole_fraction(
            ds[variable], source_kind="WACCM", target_unit=target_unit,
            context=f"WACCM {variable} {path.name}",
        )
        source_units = source.attrs["source_units"]
        conversion = source.attrs["conversion"]
        field = log_pressure_interpolate(source, hybrid_mid_pressure(ds), CHEM_PLEV)
        field = select_latitude(field, 60.0, 82.0)
        if "lon" in field.dims:
            field = field.mean("lon", skipna=True)
        polar = cosine_latitude_mean(field, 60.0, 82.0).transpose("time", "plev").load()
        dates = date_int(ds)
    polar = polar.assign_coords(date=("time", dates))
    polar.attrs.update(units=target_unit, source_units=source_units, conversion=conversion)
    return polar

def waccm_tmin(path):
    with xr.open_dataset(path, decode_times=True, chunks={"time": 12}) as ds:
        if "PS" in ds:
            field = log_pressure_interpolate(ds["T"], hybrid_mid_pressure(ds), CHEM_PLEV)
        else:
            field = ds["T"]
            pressure_name = "plev" if "plev" in field.coords else "lev"
            pressure = np.asarray(field[pressure_name].values, dtype=float)
            if np.nanmax(pressure) > 2000.0:
                pressure = pressure / 100.0
            field = field.assign_coords({pressure_name: pressure}).interp({pressure_name: CHEM_PLEV})
            if pressure_name != "plev":
                field = field.rename({pressure_name: "plev"})
        field = select_latitude(field.mean("lon", skipna=True), 60.0, 82.0)
        result = field.min("lat", skipna=True).transpose("time", "plev").load()
        dates = date_int(ds)
    return result.assign_coords(date=("time", dates))

def oct_sep(previous, current, event_year):
    merged = xr.concat([previous, current], dim="time")
    years = np.asarray(merged.date.values, dtype=int) // 10000
    md = noleap_index(merged.date.values)
    keep = (((years == event_year - 1) & (md >= OCTOBER_START)) |
            ((years == event_year) & (md < OCTOBER_START))) & (md >= 0)
    output = merged.isel(time=np.flatnonzero(keep)).sortby("date")
    if output.sizes["time"] != 365:
        raise ValueError(f"Incomplete Oct--Sep season ending in {event_year}")
    return output

def merra_tmin_event():
    pieces = []
    for year in (2019, 2020):
        path = PREPROCESSED_ROOT / "MERRA2_Processed" / "T" / f"MERRA2.T.{year}.nc"
        with xr.open_dataset(path, decode_times=True, chunks={"time": 12}) as ds:
            field = select_latitude(ds["T"], 60.0, 82.0)
            pressure_name = "lev" if "lev" in field.coords else "plev"
            pressure = np.asarray(field[pressure_name].values, dtype=float)
            if np.nanmax(pressure) > 2000.0:
                pressure = pressure / 100.0
            field = field.assign_coords({pressure_name: pressure}).interp({pressure_name: CHEM_PLEV})
            if pressure_name != "plev":
                field = field.rename({pressure_name: "plev"})
            pieces.append(field.mean("lon", skipna=True).min("lat", skipna=True).load())
    combined = xr.concat(pieces, dim="time").sortby("time").sel(time=slice("2019-10-01", "2020-09-30"))
    dates = pd.DatetimeIndex(pd.to_datetime(combined.time.values))
    combined = combined.isel(time=np.flatnonzero(~((dates.month == 2) & (dates.day == 29))))
    if combined.sizes["time"] != 365:
        raise RuntimeError("MERRA-2 chemistry temperature context must be one no-leap event year")
    return combined.transpose("time", "plev")


## Daytime MLS ClO and WACCM ClOx

The WACCM ClOx climatology is explicitly and exclusively the established 200-file `CO2x1SmidEmin_yBWCN_timefixed` source. Its path and count are recorded in output metadata.

Inputs: The read-only archive or schema-validated staging paths explicitly opened in the following code cell.

Outputs: The in-memory object(s) and atomic staging product(s) explicitly named in the following code cell; setup-only cells emit no file.

Method: Apply the scientific definition stated above, enforce its declared dimensions/counts/parameters, validate any temporary output, then atomically install it without modifying a source file.


In [ ]:
mls_clo = mls_event_anomaly("ClO", "ClO PressureZM Day", "ppbv")

# The manuscript/model ClOx climatology is the explicitly documented 200-year source.
clox_root = PREPROCESSED_ROOT / "CO2x1SmidEmin_yBWCN_timefixed" / "CLOX"
clox_paths = sorted(clox_root.glob("CO2x1SmidEmin_yBWCN.cam.h1.*.CLOX.nc"))
if len(clox_paths) != 200:
    raise RuntimeError(f"ClOx climatology source must contain exactly 200 annual files; found {len(clox_paths)}")
clox_model_years = [parse_year(path) for path in clox_paths]
if len(set(clox_model_years)) != 200:
    raise RuntimeError("ClOx climatology source must contain 200 unique model years")
clox_years, clox_source_units, clox_conversions = [], set(), set()
for path in clox_paths:
    with xr.open_dataset(path, decode_times=False) as ds:
        dates = date_int(ds)
        calendar_day = noleap_index(dates)
        if (
            dates.size != 365 or np.unique(dates).size != 365
            or np.any(np.diff(dates) <= 0)
            or not np.array_equal(calendar_day, np.arange(365))
        ):
            raise RuntimeError(
                f"ClOx climatology file lacks a strictly ordered no-leap Jan--Dec calendar: {path.name}"
            )
        field = convert_mole_fraction(
            ds["CLOX"], source_kind="WACCM", target_unit="ppbv",
            context=f"ClOx climatology {path.name}",
        )
        clox_source_units.add(field.attrs["source_units"])
        clox_conversions.add(field.attrs["conversion"])
        field = select_latitude(field, 60.0, 82.0)
        pressure_name = "lev" if "lev" in field.coords else "plev"
        pressure = np.asarray(field[pressure_name].values, dtype=float)
        if np.nanmax(pressure) > 2000.0:
            pressure = pressure / 100.0
        field = field.assign_coords({pressure_name: pressure}).interp({pressure_name: CHEM_PLEV})
        if pressure_name != "plev":
            field = field.rename({pressure_name: "plev"})
        if "lon" in field.dims:
            field = field.mean("lon", skipna=True)
        polar = cosine_latitude_mean(field, 60.0, 82.0).transpose("time", "plev").load()
    clox_years.append(np.asarray(polar.values, dtype=float))
if len(clox_source_units) != 1 or len(clox_conversions) != 1:
    raise RuntimeError("ClOx climatology source units/conversion changed across 200 years")
clox_calendar_clim = np.nanmean(np.stack(clox_years), axis=0)
clox_clim = np.concatenate([clox_calendar_clim[OCTOBER_START:], clox_calendar_clim[:OCTOBER_START]], axis=0)

bwcn_root = PREPROCESSED_ROOT / "BWCN"
clox_event = oct_sep(
    waccm_polar(bwcn_root / "CLOX" / "BWCN.cam.h3.0007.CLOX.nc", "CLOX", "ppbv"),
    waccm_polar(bwcn_root / "CLOX" / "BWCN.cam.h3.0008.CLOX.nc", "CLOX", "ppbv"), 8,
)
clox_anomaly = np.asarray(clox_event.values) - clox_clim
merra_t = merra_tmin_event()
waccm_t = oct_sep(
    waccm_tmin(bwcn_root / "T" / "BWCN.cam.h3.0007.T.nc"),
    waccm_tmin(bwcn_root / "T" / "BWCN.cam.h3.0008.T.nc"), 8,
)

clox_output = xr.Dataset(
    {
        "mls_clo_anomaly_ppbv": (("season_day", "mls_pressure_hpa"), np.asarray(mls_clo.values)),
        "waccm_clox_anomaly_ppbv": (("season_day", "waccm_pressure_hpa"), clox_anomaly),
        "merra2_tmin_k": (("season_day", "merra2_pressure_hpa"), np.asarray(merra_t.values)),
        "waccm_tmin_k": (("season_day", "waccm_t_pressure_hpa"), np.asarray(waccm_t.values)),
    },
    coords={
        "season_day": np.arange(365), "mls_pressure_hpa": mls_clo.plev.values,
        "waccm_pressure_hpa": clox_event.plev.values, "merra2_pressure_hpa": merra_t.plev.values,
        "waccm_t_pressure_hpa": waccm_t.plev.values,
    },
    attrs={
        "product_version": PRODUCT_VERSION, "method": "unsmoothed calendar-day anomalies",
        "latitude_band": "60--82N", "pressure_range_hpa": "1--100",
        "mls_climatology": "15 complete winters 2004/05--2018/19",
        "waccm_clox_climatology": str(clox_root), "waccm_clox_climatology_years": 200,
        "waccm_clox_climatology_model_years": ",".join(f"{year:04d}" for year in clox_model_years),
        "mls_clo_source_units": mls_clo.attrs["source_units"],
        "mls_clo_conversion": mls_clo.attrs["conversion"],
        "mls_calendar_gate": mls_clo.attrs["calendar_gate"],
        "mls_baseline_month_day_count": int(mls_clo.attrs["baseline_month_day_count"]),
        "mls_event_month_day_count": int(mls_clo.attrs["event_month_day_count"]),
        "waccm_clox_event_source_units": clox_event.attrs["source_units"],
        "waccm_clox_event_conversion": clox_event.attrs["conversion"],
        "waccm_clox_climatology_source_units": next(iter(clox_source_units)),
        "waccm_clox_climatology_conversion": next(iter(clox_conversions)),
        "waccm_clox_climatology_calendar": "200 individually validated ordered 365-day no-leap years",
        "waccm_clox_climatology_case_note": "200-year EXTR source is distinct from target BWCN case",
    },
)
for coordinate in (
    "mls_pressure_hpa", "waccm_pressure_hpa",
    "merra2_pressure_hpa", "waccm_t_pressure_hpa",
):
    clox_output[coordinate].attrs.update(units="hPa", positive="down")
write_netcdf_atomic(
    clox_output, product_path("chemistry", "clo_clox.nc"),
    required_vars={
        "mls_clo_anomaly_ppbv": ("season_day", "mls_pressure_hpa"),
        "waccm_clox_anomaly_ppbv": ("season_day", "waccm_pressure_hpa"),
        "merra2_tmin_k": ("season_day", "merra2_pressure_hpa"),
        "waccm_tmin_k": ("season_day", "waccm_t_pressure_hpa"),
    }, exact_sizes={"season_day": 365}, required_attrs={
        "waccm_clox_climatology_years": 200,
        "mls_clo_source_units": "vmr", "waccm_clox_event_source_units": "mol/mol",
        "waccm_clox_climatology_source_units": "mol/mol",
        "mls_baseline_month_day_count": 15, "mls_event_month_day_count": 1,
    },
    overwrite=OVERWRITE,
)


## MLS and WACCM H2O

The former hand-picked year exclusions are removed. The H2O climatology uses every complete free-running Oct--Sep season available from the strict manifest except the target BWCN:0008 season.

Inputs: The read-only archive or schema-validated staging paths explicitly opened in the following code cell.

Outputs: The in-memory object(s) and atomic staging product(s) explicitly named in the following code cell; setup-only cells emit no file.

Method: Apply the scientific definition stated above, enforce its declared dimensions/counts/parameters, validate any temporary output, then atomically install it without modifying a source file.


In [ ]:
mls_h2o = mls_event_anomaly("H2O", "H2O PressureZM", "ppmv")
manifest = pd.read_csv(product_path("ozone", "waccm_event_manifest.csv"))
bwcn_manifest = manifest.loc[manifest.source_segment == "BWCN"].copy()
if len(bwcn_manifest) != 23:
    raise RuntimeError("H2O baseline requires the strict 23-event BWCN manifest")
bwcn_h2o_root = PREPROCESSED_ROOT / "BWCN" / "H2O"
all_paths = {parse_year(path): path for path in bwcn_h2o_root.glob("*.H2O.nc")}
wanted = set(bwcn_manifest.model_year.astype(int))
cache = {
    year: waccm_polar(all_paths[year], "H2O", "ppmv")
    for year in sorted(wanted | {year - 1 for year in wanted}) if year in all_paths
}

baseline_profiles, baseline_ids = [], []
target_profile = None
for row in bwcn_manifest.itertuples(index=False):
    previous = cache.get(int(row.model_year) - 1)
    current = cache.get(int(row.model_year))
    if previous is None or current is None:
        continue
    try:
        season = oct_sep(previous, current, int(row.model_year))
    except ValueError:
        continue
    if row.event_id == "BWCN:0008":
        target_profile = season
    else:
        baseline_profiles.append(np.asarray(season.values, dtype=float))
        baseline_ids.append(row.event_id)
if target_profile is None:
    raise RuntimeError("BWCN:0008 H2O target season is missing")
if not baseline_profiles:
    raise RuntimeError("No complete target-excluded free-running H2O seasons")
if "BWCN:0008" in baseline_ids or len(set(baseline_ids)) != len(baseline_ids):
    raise RuntimeError("H2O baseline IDs must be unique and strictly target-excluded")
h2o_climatology = np.nanmean(np.stack(baseline_profiles), axis=0)
h2o_anomaly = np.asarray(target_profile.values) - h2o_climatology

merra_t = merra_tmin_event()
waccm_t = oct_sep(
    waccm_tmin(PREPROCESSED_ROOT / "BWCN" / "interpolated" / "T" / "BWCN.cam.h3.0007.T.nc"),
    waccm_tmin(PREPROCESSED_ROOT / "BWCN" / "interpolated" / "T" / "BWCN.cam.h3.0008.T.nc"), 8,
)
h2o_output = xr.Dataset(
    {
        "mls_h2o_anomaly_ppmv": (("season_day", "mls_pressure_hpa"), np.asarray(mls_h2o.values)),
        "waccm_h2o_anomaly_ppmv": (("season_day", "waccm_pressure_hpa"), h2o_anomaly),
        "merra2_tmin_k": (("season_day", "merra2_pressure_hpa"), np.asarray(merra_t.values)),
        "waccm_tmin_k": (("season_day", "waccm_t_pressure_hpa"), np.asarray(waccm_t.values)),
    },
    coords={
        "season_day": np.arange(365), "mls_pressure_hpa": mls_h2o.plev.values,
        "waccm_pressure_hpa": target_profile.plev.values, "merra2_pressure_hpa": merra_t.plev.values,
        "waccm_t_pressure_hpa": waccm_t.plev.values,
    },
    attrs={
        "product_version": PRODUCT_VERSION, "method": "unsmoothed calendar-day anomalies",
        "latitude_band": "60--82N", "pressure_range_hpa": "1--100",
        "mls_climatology": "15 complete winters 2004/05--2018/19",
        "waccm_h2o_climatology": "all complete BWCN restart-source Oct--Sep seasons except BWCN:0008",
        "waccm_h2o_baseline_season_count": len(baseline_profiles),
        "waccm_h2o_baseline_event_ids": ",".join(baseline_ids),
        "waccm_h2o_manifest": str(product_path("ozone", "waccm_event_manifest.csv")),
        "target_excluded": "True",
        "target_event_id": "BWCN:0008",
        "mls_h2o_source_units": mls_h2o.attrs["source_units"],
        "mls_h2o_conversion": mls_h2o.attrs["conversion"],
        "mls_calendar_gate": mls_h2o.attrs["calendar_gate"],
        "mls_baseline_month_day_count": int(mls_h2o.attrs["baseline_month_day_count"]),
        "mls_event_month_day_count": int(mls_h2o.attrs["event_month_day_count"]),
        "waccm_h2o_source_units": target_profile.attrs["source_units"],
        "waccm_h2o_conversion": target_profile.attrs["conversion"],
    },
)
for coordinate in (
    "mls_pressure_hpa", "waccm_pressure_hpa",
    "merra2_pressure_hpa", "waccm_t_pressure_hpa",
):
    h2o_output[coordinate].attrs.update(units="hPa", positive="down")
write_netcdf_atomic(
    h2o_output, product_path("chemistry", "h2o.nc"),
    required_vars={
        "mls_h2o_anomaly_ppmv": ("season_day", "mls_pressure_hpa"),
        "waccm_h2o_anomaly_ppmv": ("season_day", "waccm_pressure_hpa"),
        "merra2_tmin_k": ("season_day", "merra2_pressure_hpa"),
        "waccm_tmin_k": ("season_day", "waccm_t_pressure_hpa"),
    }, exact_sizes={"season_day": 365}, required_attrs={
        "target_excluded": "True", "target_event_id": "BWCN:0008",
        "mls_h2o_source_units": "vmr", "waccm_h2o_source_units": "mol/mol",
        "waccm_h2o_baseline_season_count": len(baseline_profiles),
        "mls_baseline_month_day_count": 15, "mls_event_month_day_count": 1,
    },
    overwrite=OVERWRITE,
)
